In [14]:
import numpy as np
import pandas as pd
import warnings
import light_curve as lc
import lightgbm as lgb
import optuna

from tqdm import tqdm, TqdmWarning
from datetime import datetime
from pathlib import Path
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from typing import Literal, Callable, Any


warnings.filterwarnings("ignore", category=TqdmWarning)

In [15]:
Type = Literal["train", "test"]
Split = Literal["split_01", "split_02", "split_03", "split_04", "split_05", "split_06", "split_07", "split_08", "split_09", "split_10", "split_11", "split_12", "split_13", "split_14", "split_15", "split_16", "split_17", "split_18", "split_19", "split_20"]


EPS = np.finfo(float).eps


def now() -> str:
    return datetime.now().astimezone().strftime("%Y%m%d-%H%M%S-%z")

### Data Loading

In [16]:
def load_log_df(type: Type, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{type}_log.parquet", **kwargs)

def load_flc_df(type: Type, split: Split, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet", **kwargs)


def __ingest_dfs(type: Type):
    log_df = pd.read_csv(f"../artifacts/kaggle/{type}_log.csv", index_col="object_id")
    log_df.to_parquet(f"../artifacts/kaggle/{type}_log.parquet")

    splits = sorted(log_df["split"].unique())
    for split in splits:
        flc_df = pd.read_csv(f"../artifacts/kaggle/{split}/{type}_full_lightcurves.csv")
        flc_df.to_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet")


__ingest_dfs(type="train")
__ingest_dfs(type="test")

### Feature Engineering

In [17]:
def load_all_feats_df(type: Type, **kwargs):
    return pd.concat([pd.read_parquet(filepath, **kwargs) for filepath in Path("../artifacts/feats").rglob(f"{type}_feats.parquet")])

def load_feats_df(type: Type, split: Split, **kwargs):
    return pd.read_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet", **kwargs)


def __build_and_ingest_feats(type: Type):
    log_df = load_log_df(type=type)

    with tqdm(total=len(log_df), unit="obj") as pb:
        for split, log_sub_df in log_df.groupby("split"):
            pb.set_description(f"Building features for `{split}` (`{type}`)")

            feats_buf = []

            for obj_id, log_row in log_sub_df.iterrows():
                flc_df = load_flc_df(type=type, split=split, filters=[("object_id", "==", obj_id)]) # pyright: ignore[reportArgumentType]
                feats = __build_feats_for_obj(log_row, flc_df)
                feats_buf.append(feats)

                pb.update()

            Path(f"../artifacts/feats/{split}").mkdir(parents=True, exist_ok=True)

            pd.DataFrame(feats_buf, index=log_sub_df.index).to_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet")


# TODO Add features acquired from domain knowledge
def __build_feats_for_obj(log_row: pd.Series, flc_df: pd.DataFrame) -> dict:
    # Could reorder?
    __de_extinct(log_row, flc_df)

    flc_df["Flux_ratio"] = flc_df["Flux"] / flc_df["Flux_err"]

    feats = log_row.to_dict()

    feats.update(__build_stats_feats_for_obj(flc_df))
    feats.update(__build_lc_feats_for_obj(flc_df))
    feats.update(__build_domain_feats_for_obj(flc_df))

    return feats


def __build_stats_feats_for_obj(flc_df: pd.DataFrame) -> dict:
    feats = {}

    pivot_flc_df = flc_df.pivot_table(index="Time (MJD)", columns="Filter", values=["Flux", "Flux_err", "Flux_ratio"])

    for agg_name, agg in __STATS_AGGS.items():
        for feat in ["Flux", "Flux_err", "Flux_ratio"]:
            feats[f"{feat}_{agg_name}"] = agg(flc_df[feat])
            
            for filter in __FILTERS:
                if filter in pivot_flc_df.columns:
                    feats[f"{feat}_{agg_name}_{filter}"] = agg(pivot_flc_df[(feat, filter)]) # pyright: ignore[reportCallIssue]

    return feats


__STATS_AGGS: dict[str, Callable[[pd.Series], Any]] = {
    "mean": np.mean,
    "std": np.std,
    "min": np.min,
    "max": np.max,
    "median": np.median,
    "q25": lambda feats: feats.quantile(0.25),
    "q75": lambda feats: feats.quantile(0.75),
}
__FILTERS = ["u", "g", "r", "i", "z", "y"]


def __build_lc_feats_for_obj(flc_df: pd.DataFrame) -> dict:   
    feats = {}

    def lc_fe(df: pd.DataFrame):
        return __LC_FE(df.index.to_numpy(dtype=np.float64), df["Flux"].to_numpy(), df["Flux_err"].to_numpy()) # pyright: ignore[reportCallIssue]

    # TODO Try optimizing
    flc_fin_df = flc_df[(np.isfinite(flc_df.index) & np.isfinite(flc_df["Flux"]) & np.isfinite(flc_df["Flux_err"]))]
    
    if len(flc_fin_df) >= __LC_FE_MIN_NROWS:
        lc_feats = lc_fe(flc_fin_df)

        for feat_name, feat in zip(__LC_FE.names, lc_feats): # pyright: ignore[reportAttributeAccessIssue]
            feats[feat_name] = feat

    for filter in __FILTERS:
        flc_fin_sub_df = flc_fin_df.loc[flc_df["Filter"] == filter]

        if len(flc_fin_sub_df) >= __LC_FE_MIN_NROWS:
            lc_feats = lc_fe(flc_fin_sub_df)

            for feat_name, feat in zip(__LC_FE.names, lc_feats): # pyright: ignore[reportAttributeAccessIssue]
                feats[f"{feat_name}_{filter}"] = feat

    return feats


def __build_domain_feats_for_obj(flc_df: pd.DataFrame) -> dict:
    return {}


__LC_FE = lc.Extractor(
    lc.LinearFit(), # pyright: ignore[reportArgumentType]
    lc.StetsonK(), # pyright: ignore[reportArgumentType]
    lc.Amplitude(), # pyright: ignore[reportArgumentType]
    lc.BeyondNStd(), # pyright: ignore[reportArgumentType]
    lc.Skew(), # pyright: ignore[reportArgumentType]
    lc.Kurtosis(), # pyright: ignore[reportArgumentType]
)
__LC_FE_MIN_NROWS = 4


# TODO Consider extinction.fitzpatrick99
def __de_extinct(log_row: pd.Series, flc_sub_df: pd.DataFrame):
    r_λ = flc_sub_df["Filter"].map({
        "u": 4.81,
        "g": 3.64,
        "r": 2.70,
        "i": 2.06,
        "z": 1.58,
        "y": 1.31
    })

    c_λ = np.pow(10, 0.4 * r_λ * log_row["EBV"])

    flc_sub_df["Flux"] *= c_λ
    flc_sub_df["Flux_err"] *= c_λ


__build_and_ingest_feats(type="train")
__build_and_ingest_feats(type="test")

Building features for `split_20` (`test`): 100%|██████████| 7135/7135 [01:20<00:00, 89.14obj/s]


### Data Cleaning

In [18]:
train_df = load_all_feats_df(type="train")

X = train_df.drop(columns=["SpecType", "English Translation", "split", "target"])
y = train_df["target"]

X

,Z,Z_err,EBV,Flux_mean,Flux_err_mean,Flux_ratio_mean,Flux_std,Flux_err_std,Flux_ratio_std,Flux_min,...,skew_z,kurtosis_z,linear_fit_slope_y,linear_fit_slope_sigma_y,linear_fit_reduced_chi2_y,stetson_K_y,amplitude_y,beyond_1_std_y,skew_y,kurtosis_y
object_id,,,,,,,,,,,,,,,,,,,,,
Dornhoth_fervain_onodrim,3.0490,NaN,0.110,1.168616,0.622708,4.149514,5.789107,0.585147,18.428182,-3.147488,...,3.328534,11.318189,0.018636,0.026917,3.581662,0.828991,4.923171,0.090909,1.525677,2.634059
Dornhoth_galadh_ylf,0.4324,NaN,0.058,0.429695,0.587684,1.024884,1.488888,0.561601,3.150463,-1.873722,...,3.127142,12.145236,-0.041316,0.005185,5.486354,0.535131,7.036898,0.103448,2.983446,11.688474
Elrim_melethril_thul,0.4673,NaN,0.577,5.218302,1.305302,6.251681,6.401231,0.998163,6.124101,-12.840542,...,0.046864,-1.455541,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ithil_tobas_rodwen,0.6946,NaN,0.012,0.385203,0.428335,1.657185,0.875471,0.472227,2.612474,-7.753266,...,0.519980,0.730480,0.000576,0.000398,1.351590,0.798893,6.592584,0.226087,-0.967255,3.308255
Mirion_adar_Druadan,0.4161,NaN,0.058,0.260234,0.482768,0.888452,1.276400,0.461308,4.496637,-3.282238,...,-0.159916,0.382455,-0.018741,0.008021,1.603544,0.814502,2.912294,0.428571,-0.802999,-0.558952
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tinnu_gellui_tathar,0.8898,NaN,0.042,0.492624,0.555184,1.359930,1.062509,0.501120,2.312394,-1.493477,...,0.679493,0.152494,0.017091,0.006070,1.234541,0.863899,4.128050,0.250000,1.147746,1.782013
uir_heleg_corf,0.9598,NaN,0.042,0.422353,0.523150,1.416870,1.614420,0.501490,5.103314,-5.607301,...,3.484227,14.988520,-0.013627,0.007295,1.095413,0.825562,4.086785,0.166667,-1.715446,4.783379
uir_rhosc_law,0.1543,NaN,0.024,0.392762,0.518548,1.253894,1.110451,0.540616,2.772301,-2.854501,...,1.157686,0.265259,-0.015444,0.004689,1.915739,0.751998,4.044818,0.290323,1.061887,1.467398


### Aggressive Hyperparameter Tuning with F1 Threshold Optimization

In [ ]:
def __objective(trial: optuna.Trial):
    # Hyperparameters Suggestion
    lgbm_params = {
        "objective": "binary",
        "metric": "average_precision",
        # "metric": "binary_logloss",
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "dart", "rf", "goss"]),
        "n_jobs": -1,
        "verbosity": -1,
        # "is_unbalance": True,
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", low=1.0, high=200.0),

        "device": "gpu",
        # "max_bin": trial.suggest_categorical("max_bin", [63, 255]),

        # Tree Structure
        "num_leaves": trial.suggest_int("num_leaves", low=16, high=256),
        # "max_depth": trial.suggest_int("max_depth", low=6, high=16),
        "max_depth": -1,
        "min_child_samples": trial.suggest_int("min_child_samples", low=5, high=100),

        # Learning Speed
        "learning_rate": trial.suggest_float("learning_rate", low=0.005, high=0.1, log=True), # Slower learning rate leads to better accuracy
        "n_estimators": trial.suggest_int("n_estimators", low=100, high=10000),

        # Regularization
        "reg_alpha": trial.suggest_float("reg_alpha", low=1e-4, high=1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", low=1e-4, high=1.0, log=True),
        # "min_split_gain": trial.suggest_float("min_split_gain", low=1e-6, high=1.0, log=True),

        # Sampling
        "subsample": trial.suggest_float("subsample", low=0.5, high=0.95), # Stochasticity helps generalization
        # "subsample_freq": trial.suggest_int("subsample_freq", low=1, high=5),
        "colsample_bytree": trial.suggest_float("colsample_bytree", low=0.5, high=0.95),
    }

    # Cross-Validation
    skf = StratifiedKFold(n_splits=10, random_state=42, shuffle=True)
    oof_preds = np.zeros(len(X))

    for (train_idx, val_idx) in skf.split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        lgbm = lgb.LGBMClassifier(**lgbm_params)
        lgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="f1", callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            # optuna.integration.LightGBMPruningCallback(trial, "f1"),
        ])

        oof_preds[val_idx] = lgbm.predict_proba(X_val)[:, 1] # type: ignore
    
    # Dynamic Threshold Tuning
    thresholds = np.linspace(0.01, 0.99, 2000)

    f1_scores = np.array([f1_score(y, (oof_preds > threshold).astype(int)) for threshold in thresholds])
    idx = f1_scores.argmax()

    trial.set_user_attr("best_threshold", thresholds[idx])
    return f1_scores[idx]


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(n_startup_trials=44, seed=42, multivariate=True))
study.optimize(__objective, n_trials=200, n_jobs=-1, show_progress_bar=True) # pyright: ignore[reportArgumentType]

print(f"Study concluded with: best F1 score `{study.best_value:.4f}`; best threshold `{study.best_trial.user_attrs["best_threshold"]:.2f}`; best hyperparameters: `{study.best_params}`")

c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2025-12-11 18:51:26,150] A new study created in memory with name: no-name-138f52d9-86df-4484-9074-79bd0bdb2e5b
  0%|          | 0/200 [00:00<?, ?it/s]c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\l

[I 2025-12-11 18:52:50,996] Trial 13 finished with value: 0.23586744639376217 and parameters: {'boosting_type': 'rf', 'scale_pos_weight': 19.483817700885762, 'num_leaves': 111, 'min_child_samples': 97, 'learning_rate': 0.060158763700839044, 'n_estimators': 2026, 'reg_alpha': 0.0001754564854395112, 'reg_lambda': 0.0001462566073241255, 'subsample': 0.8654529690093886, 'colsample_bytree': 0.9277214403958228}. Best is trial 13 with value: 0.23586744639376217.


c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
Best trial: 0. Best value: 0.25998:   1%|          | 2/200 [01:54<2:52:09, 52.17s/it]  

[I 2025-12-11 18:53:20,289] Trial 0 finished with value: 0.2599795291709314 and parameters: {'boosting_type': 'rf', 'scale_pos_weight': 102.47559389024556, 'num_leaves': 111, 'min_child_samples': 49, 'learning_rate': 0.07877148134949151, 'n_estimators': 3682, 'reg_alpha': 0.0008382518027313924, 'reg_lambda': 0.00911574213830404, 'subsample': 0.6756771753215685, 'colsample_bytree': 0.9349133584314486}. Best is trial 0 with value: 0.2599795291709314.


c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
Best trial: 12. Best value: 0.442177:   2%|▏         | 3/200 [02:30<2:26:59, 44.77s/it]

...

c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
c:\Users\nguye\AppDa

[I 2025-12-11 20:24:27,490] Trial 63 finished with value: 0.4234527687296417 and parameters: {'boosting_type': 'goss', 'scale_pos_weight': 82.85269358338033, 'num_leaves': 253, 'min_child_samples': 28, 'learning_rate': 0.07853106651652249, 'n_estimators': 2820, 'reg_alpha': 0.013358884953763457, 'reg_lambda': 0.7193516147127974, 'subsample': 0.6298717513514515, 'colsample_bytree': 0.8577917656000942}. Best is trial 3 with value: 0.4899135446685879.


c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
Best trial: 3. Best 

[I 2025-12-11 20:27:00,994] Trial 56 finished with value: 0.42191780821917807 and parameters: {'boosting_type': 'goss', 'scale_pos_weight': 35.92809859857702, 'num_leaves': 111, 'min_child_samples': 15, 'learning_rate': 0.04235824831800386, 'n_estimators': 1472, 'reg_alpha': 0.05608083488994835, 'reg_lambda': 0.0004201927680376044, 'subsample': 0.6819696238196088, 'colsample_bytree': 0.8220150323854356}. Best is trial 3 with value: 0.4899135446685879.


c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")
c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.14\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")


In [ ]:
lgbm_params = study.best_params
lgbm_params.update({
    "objective": "binary",
    "metric": "average_precision",
    "boosting_type": "gbdt",
    "n_jobs": -1,
    "verbosity": -1,
})

threshold = study.best_trial.user_attrs["best_threshold"]

### Feature Importance Analysis

### Inference

In [ ]:
test_df = load_all_feats_df(type="test")

X_test = test_df.drop(columns=["SpecType", "English Translation", "split"])

X_test

In [ ]:
lgbm = lgb.LGBMClassifier(**lgbm_params)
lgbm.fit(X, y)

test_probs = lgbm.predict_proba(X_test)[:, 1] # type: ignore
test_preds = (test_probs > threshold).astype(int)

In [ ]:
Path("../artifacts/preds").mkdir(parents=True, exist_ok=True)

test_preds_df = pd.DataFrame({"object_id": X_test.index, "target": test_preds})
test_preds_df.to_csv(f"../artifacts/preds/submission-{now()}.csv", index=False)

In [ ]:
# TODO
# Assuming GB, no imputation, no scaling, only minimal cleaning.
# Plot f1 score changes over trials & hyperparams
# Plot feature importance
# Set seeds to 67